# Diabetic Retinopathy Severity Classification
EfficientNet-B0 fine-tuned on the [APTOS 2019 Blindness Detection](https://www.kaggle.com/c/aptos2019-blindness-detection/data) dataset.

**Before running:** Make sure the runtime is set to GPU — Runtime → Change runtime type → T4 GPU.

## 1. Git Setup

In [40]:
!git config --global user.name "PaarthP89"
!git config --global user.email "paarthp89@gmail.com"

## 2. Clone Repository

In [2]:
import os

REPO_URL = "https://github.com/noteesh/Diabetic-Retinopathy-Severity-Classification.git"
REPO_DIR = "Diabetic-Retinopathy-Severity-Classification"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

Cloning into 'Diabetic-Retinopathy-Severity-Classification'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 56 (delta 20), reused 44 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 2.92 MiB | 16.35 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/Diabetic-Retinopathy-Severity-Classification
Working directory: /content/Diabetic-Retinopathy-Severity-Classification


## 3. Install Dependencies

In [3]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.6 MB/s eta 0:00:00


## 4. Just take the L and import the data yourself


its google drive - share the dataset zip to your google drive account, then add
a shortcut to My Drive, then you should be good.

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs('data', exist_ok=True)
!unzip -q "/content/drive/MyDrive/aptos2019-blindness-detection.zip" -d data/
print("Files:", os.listdir('data'))

Mounted at /content/drive
Files: ['train.csv', 'train_images', 'sample_submission.csv', 'test_images', 'test.csv']


## 6. Resize Photos cause it's taking way too long to train


In [5]:
from PIL import Image
from pathlib import Path
import os

src = Path('data/train_images')
dst = Path('data/train_images_512')
dst.mkdir(exist_ok=True)

for f in src.glob('*.png'):
    img = Image.open(f).convert('RGB')
    img = img.resize((512, 512), Image.LANCZOS)
    img.save(dst / f.name, 'PNG', optimize=False, compress_level=1)

print('Done')

Done


In [11]:
import cv2
import numpy as np
from PIL import Image
from pathlib import Path

src = Path('data/train_images')
dst = Path('data/train_images_380')
dst.mkdir(exist_ok=True)

def process(img_np, size=380, sigmaX=10):
    img_np = np.array(Image.fromarray(img_np).resize((size, size), Image.LANCZOS))
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        img_np = img_np[y:y+h, x:x+w]

    img_np = np.array(Image.fromarray(img_np).resize((size, size), Image.LANCZOS))
    blurred = cv2.GaussianBlur(img_np, (0, 0), sigmaX)
    img_np = np.clip(cv2.addWeighted(img_np, 4, blurred, -4, 128), 0, 255).astype(np.uint8)
    return Image.fromarray(img_np)

files = list(src.glob('*.png'))
for i, f in enumerate(files):
    img = Image.open(f).convert('RGB')
    processed_img = process(np.array(img))
    processed_img.save(dst / f.name, 'PNG', optimize=False, compress_level=1)

    if (i + 1) % 500 == 0:
        print(f'{i+1}/{len(files)}')

print('Done')

500/3662
1000/3662
1500/3662
2000/3662
2500/3662
3000/3662
3500/3662
Done


In [26]:
%%writefile src/improved_model.py
import torch.nn as nn
from torchvision import models

class ImprovedDRClassifier(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3, freeze_backbone=False):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        self.model = models.efficientnet_b0(weights=weights)

        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=dropout, inplace=True),
            nn.Linear(in_features, num_classes),
        )

        for param in self.model.features.parameters():
            param.requires_grad = not freeze_backbone

    def forward(self, x):
        return self.model(x)

    def get_last_conv_layer(self):
        return self.model.features[-1]

Overwriting src/improved_model.py


## 7. Train the Model

T4 handles batch size 16 comfortably at 380px. Ben Graham preprocessing (`--preprocess`) is applied at load time.

In [53]:
!git log --oneline -3
!git push origin main

23ac5db (HEAD -> main, origin/main, origin/HEAD) New training with B0 Ben Graham
bd2226f new ipynb changes.
2aabfde Trying to improve model.
remote: Invalid username or token. Password authentication is not supported for Git operations.
fatal: Authentication failed for 'https://github.com/noteesh/Diabetic-Retinopathy-Severity-Classification.git/'


## 8. Results

In [ ]:
from IPython.display import Image as IPyImage, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

result_plots = [
    ('results/improved_training_curves.png', 'Training Curves'),
    ('results/improved_confusion_matrix.png', 'Confusion Matrix'),
]

for path, title in result_plots:
    if os.path.exists(path):
        fig, ax = plt.subplots(figsize=(12, 5) if 'curves' in path else (8, 7))
        ax.imshow(mpimg.imread(path))
        ax.axis('off')
        ax.set_title(title, fontsize=14, pad=10)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Not found: {path}")

## 9. Save Results to Google Drive (optional)

In [ ]:
# Uncomment to mount Drive and copy results

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree('results', '/content/drive/MyDrive/DR_results', dirs_exist_ok=True)
# print("Results saved to Google Drive.")